# GHG facility emissions by state (choropleth)

Aggregate EPA GHGRP facility-level CO₂e emissions to the state level and render a simple choropleth on top of `sondio.geo.subdivisions` polygons. Requires the `sondio[geo]` extra.

In [ ]:
%pip install --quiet 'sondio[geo]>=0.1.2,<0.2' matplotlib

# sondio >= 0.1.2 auto-resolves your key from (in order):
#   1. Colab Secrets — add a secret named SONDIO_API_KEY (🔑 sidebar, toggle Notebook access)
#   2. Kaggle Secrets — add a secret named SONDIO_API_KEY (Add-ons → Secrets)
#   3. SONDIO_API_KEY environment variable (Deepnote / Hex / SageMaker / local)
#   4. ~/.sondio/config (local CLI users)
# Grab a free key at https://sondio.io/developers
import sondio
print(f"sondio {sondio.__version__}")

In [ ]:
# All reporting facilities, latest year the API returns. Paginate — the
# dataset is ~8k facilities, well within the free-tier page cap.
fac = sondio.us.ghg.facilities(all_pages=True)
print(f"{len(fac)} facilities")
fac.head()

In [ ]:
# Aggregate CO2e by state.
by_state = (
    fac.dropna(subset=["state", "total_co2e"])
       .groupby("state", as_index=False)["total_co2e"].sum()
       .rename(columns={"total_co2e": "co2e_tons"})
       .sort_values("co2e_tons", ascending=False)
)
by_state.head(10)

In [ ]:
# Fetch state polygons (GeoDataFrame, EPSG:4326).
states = sondio.geo.subdivisions(country="US", with_geometry=True)
print(f"{len(states)} subdivisions")
states.head()

In [ ]:
# Merge emissions onto the polygons via local_code (e.g. 'TX').
states = states.merge(by_state, left_on="local_code", right_on="state", how="left")
states["co2e_tons"] = states["co2e_tons"].fillna(0)
states[["local_code", "name", "co2e_tons"]].sort_values("co2e_tons", ascending=False).head(10)

In [ ]:
import matplotlib.pyplot as plt

# Clip to CONUS so Alaska doesn't dominate the frame.
conus = states.loc[~states["local_code"].isin(["AK", "HI", "PR", "VI", "GU", "AS", "MP"])]

fig, ax = plt.subplots(figsize=(10, 6))
conus.plot(
    column="co2e_tons", ax=ax, cmap="OrRd",
    legend=True, legend_kwds={"label": "CO2e (metric tons)", "shrink": 0.6},
    edgecolor="#555", linewidth=0.4,
)
ax.set_axis_off()
ax.set_title("EPA GHGRP reported CO2e by state (CONUS)")
plt.show()

## Next steps
- Filter by `industry_type` (e.g. power plants, oil & gas production).
- Swap the aggregate for `mean` or `count(facility_id)` to study intensity.
- Clip to a single state and overlay well locations for a cross-vertical view.